# Data Parallel Loading: Simulating DistributedSampler — Solution

In [ ]:
import math
import torch
from torch.utils.data import Dataset, DataLoader, Sampler

class SimpleDataset(Dataset):
    def __init__(self, N):
        self.data = torch.arange(N, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = SimpleDataset(103)

In [ ]:
class SimpleDistributedSampler(Sampler):
    """Simplified DistributedSampler: shards a dataset across ranks."""

    def __init__(self, dataset, num_replicas: int, rank: int,
                 shuffle: bool = True, seed: int = 0):
        self.dataset = dataset
        self.num_replicas = num_replicas
        self.rank = rank
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.num_samples = math.ceil(len(dataset) / num_replicas)
        self.total_size = self.num_samples * num_replicas

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.num_samples

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(len(self.dataset), generator=g).tolist()
        else:
            indices = list(range(len(self.dataset)))

        # Pad to total_size by repeating from the start (the list is repeated
        # as often as needed when there are more replicas than samples)
        padding = self.total_size - len(indices)
        if padding > 0:
            indices += (indices * math.ceil(padding / len(indices)))[:padding]

        # Shard: this rank takes every num_replicas-th index
        indices = indices[self.rank::self.num_replicas]
        assert len(indices) == self.num_samples
        return iter(indices)


# Demo: show how indices are split across 4 ranks
for r in range(4):
    s = SimpleDistributedSampler(ds, num_replicas=4, rank=r, shuffle=False)
    idx = list(s)
    print(f"Rank {r}: {len(idx)} samples, first 5: {idx[:5]}, last 5: {idx[-5:]}")

In [ ]:
# === Tests ===
num_replicas = 4

# Test 1: All ranks get the same number of samples
samplers = [SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=r, shuffle=False)
            for r in range(num_replicas)]
lengths = [len(s) for s in samplers]
assert len(set(lengths)) == 1, f"Ranks got different lengths: {lengths}"
expected_per_rank = math.ceil(len(ds) / num_replicas)
assert lengths[0] == expected_per_rank, \
    f"Expected {expected_per_rank} per rank, got {lengths[0]}"
print(f"PASSED: All {num_replicas} ranks get {lengths[0]} samples each")

# Test 2: Shards cover the dataset; the only repeats are the padding
from collections import Counter

def check_shards(samplers, n):
    flat = [int(i) for s in samplers for i in s]
    total = len(samplers) * math.ceil(n / len(samplers))
    assert len(flat) == total, f"Total indices count is wrong: {len(flat)} != {total}"
    assert set(flat) == set(range(n)), "Some dataset indices are missing (or out of range)"
    repeats = sum(c - 1 for c in Counter(flat).values())
    assert repeats == total - n, f"{repeats} indices are repeated, but only {total - n} padding repeats are expected"

check_shards(samplers, len(ds))
check_shards([SimpleDistributedSampler(SimpleDataset(100), num_replicas, r, shuffle=False)
              for r in range(num_replicas)], 100)  # divisible: strictly disjoint
print("PASSED: Shards cover the full dataset; only the padding is repeated")

# Test 3: Works as DataLoader sampler
sampler_r0 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=False)
loader = DataLoader(ds, batch_size=8, sampler=sampler_r0)
loaded = torch.cat([b for b in loader])
assert len(loaded) == expected_per_rank, f"DataLoader yielded {len(loaded)}, expected {expected_per_rank}"
print("PASSED: Works with DataLoader")

# Test 4: Deterministic shuffling — same seed+epoch → same order
s1 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s2 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
assert list(s1) == list(s2), "Same seed should give same order"
print("PASSED: Deterministic shuffle with same seed")

# Test 5: Different epochs → different order
s3 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s3.set_epoch(1)
s4 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s4.set_epoch(2)
assert list(s3) != list(s4), "Different epochs should give different orders"
print("PASSED: Different epochs produce different shuffles")

# Test 6: With shuffling, all ranks must use the same permutation, or their shards overlap
for epoch in (0, 3):
    shuffled = [SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=r, shuffle=True, seed=7)
                for r in range(num_replicas)]
    for s in shuffled:
        s.set_epoch(epoch)
    check_shards(shuffled, len(ds))
print("PASSED: Shuffled shards cover the dataset, only the padding repeated (same permutation on every rank)")

# Test 7: More ranks than samples - every rank still gets one index
tiny = SimpleDataset(3)
tiny_samplers = [SimpleDistributedSampler(tiny, num_replicas=8, rank=r, shuffle=False) for r in range(8)]
assert all(len(s) == 1 and len(list(s)) == 1 for s in tiny_samplers), "Each of 8 ranks should get 1 index of a 3-sample dataset"
check_shards(tiny_samplers, 3)
print("PASSED: Padding works when there are more ranks than samples")

print("\nAll tests passed!")

### Notes for the interview

- **Why pad instead of dropping?** Every rank must run the same number of steps: DDP all-reduces gradients every step, and a rank that runs out of data early leaves the others hanging in the collective. `drop_last=True` (in the real `DistributedSampler`) trims the tail instead of padding.
- **Padding duplicates a few samples**, so metrics computed over a distributed eval set are slightly biased unless you de-duplicate the extra `total_size - len(dataset)` samples.
- **Strided vs. contiguous shards.** `indices[rank::num_replicas]` (what PyTorch does) and contiguous blocks are equally valid after shuffling; without shuffling, strided shards give every rank a similar mix of the dataset.
- **`set_epoch` must be called on every rank, every epoch**, before the DataLoader iterator is created — the permutation is computed in `__iter__`.